In [1]:
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
from tqdm import tqdm
import pandas as pd

# config (edit as needed)
MODEL = "google/gemma-3-12b-it"
SYSTEM_PROMPT = "You are an expert at generating realistic and culturally-relevant math word problems tailored to the country."
INPUT_PATH = "data/incontext_marathi.csv"
OUTPUT_PATH_1 = "outputs/incontext_marathi_gemma.xlsx"
OUTPUT_PATH_2 = "outputs/incontext_marathi_gemma_extracted.xlsx"
BATCH_SIZE = 32


# load data, model, tokenizer
df = pd.read_csv(INPUT_PATH)
llm = LLM(model=MODEL, tensor_parallel_size=1, download_dir="/nesi/nobackup/massey04342/models", gpu_memory_utilization=0.9)
# For thinking mode (enable_thinking=True), use Temperature=0.6, TopP=0.95, TopK=20, and MinP=0. 
#DO NOT use greedy decoding, as it can lead to performance degradation and endless repetitions.
sampling = SamplingParams(temperature=0.6, top_p=0.95, top_k=20, max_tokens=5120)
tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True, cache_dir="/nesi/nobackup/massey04342/models")

# build prompts (None for empties)
prompts = []
for p in df["zeroshot_prompt"]:
    if pd.isna(p) or str(p).strip() == "":
        prompts.append(None)
    else:
        messages = [
            {
                "role": "system",
                "content": [{"type": "text", "text": SYSTEM_PROMPT}]
            },
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": p}
                ]
            }
        ]

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        prompts.append(prompt)

    
# indices & results placeholder
non_empty_indices = [i for i, p in enumerate(prompts) if p is not None]
results = [""] * len(prompts)

# batch loop (simple)
for start in tqdm(range(0, len(non_empty_indices), BATCH_SIZE), desc="Batched generation"):
    batch_indices = non_empty_indices[start:start + BATCH_SIZE]
    batch_prompts = [prompts[i] for i in batch_indices]
    try:
        outs = llm.generate(batch_prompts, sampling_params=sampling)
        for j, out in enumerate(outs):
            text = tokenizer.decode(out.outputs[0].token_ids, skip_special_tokens=True).strip("\n")
            results[batch_indices[j]] = text
    except Exception as e:
        for i in batch_indices:
            results[i] = f"[ERROR] {type(e).__name__}: {str(e)[:200]}"

# save back to dataframe
df["zeroshot_response"] = results




prompts = []
for p in df["oneshot_prompt"]:
    if pd.isna(p) or str(p).strip() == "":
        prompts.append(None)
    else:
        messages = [
            {
                "role": "system",
                "content": [{"type": "text", "text": SYSTEM_PROMPT}]
            },
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": p}
                ]
            }
        ]

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        prompts.append(prompt)

    
# indices & results placeholder
non_empty_indices = [i for i, p in enumerate(prompts) if p is not None]
results = [""] * len(prompts)

# batch loop (simple)
for start in tqdm(range(0, len(non_empty_indices), BATCH_SIZE), desc="Batched generation"):
    batch_indices = non_empty_indices[start:start + BATCH_SIZE]
    batch_prompts = [prompts[i] for i in batch_indices]
    try:
        outs = llm.generate(batch_prompts, sampling_params=sampling)
        for j, out in enumerate(outs):
            text = tokenizer.decode(out.outputs[0].token_ids, skip_special_tokens=True).strip("\n")
            results[batch_indices[j]] = text
    except Exception as e:
        for i in batch_indices:
            results[i] = f"[ERROR] {type(e).__name__}: {str(e)[:200]}"

# save back to dataframe
df["oneshot_response"] = results


df.to_excel(OUTPUT_PATH_1, index=False, engine="openpyxl")

INFO 06-11 22:12:58 [utils.py:253] non-default args: {'download_dir': '/nesi/nobackup/massey04342/models', 'disable_log_stats': True, 'model': 'google/gemma-3-12b-it'}


INFO 06-11 22:13:00 [model.py:514] Resolved architecture: Gemma3ForConditionalGeneration


INFO 06-11 22:13:00 [model.py:1661] Using max model len 131072


INFO 06-11 22:13:57 [scheduler.py:230] Chunked prefill is enabled with max_num_batched_tokens=16384.


WARNING 06-11 22:13:57 [cuda.py:244] Forcing --disable_chunked_mm_input for models with multimodal-bidirectional attention.


...

Batched generation:  50%|█████     | 1/2 [00:02<00:02,  2.41s/it]

Adding requests:   0%|          | 0/4 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/4 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Batched generation: 100%|██████████| 2/2 [00:03<00:00,  1.54s/it]

Batched generation: 100%|██████████| 2/2 [00:03<00:00,  1.67s/it]

In [20]:
import json
import re
import ast

def _extract_last_json_str(s):
    if not isinstance(s, str):
        return None
    i = s.rfind("{")
    if i == -1:
        return None
    # walk forward to find matching closing brace (handles nested braces)
    depth = 0
    end = None
    for j in range(i, len(s)):
        c = s[j]
        if c == "{":
            depth += 1
        elif c == "}":
            depth -= 1
            if depth == 0:
                end = j + 1
                break
    candidate = s[i:end] if end is not None else s[i:]  # if no closing brace, take to end
    return candidate.strip()

def _parse_loose_json(candidate):
    if candidate is None:
        return None
    # 1) Try strict JSON
    try:
        return json.loads(candidate)
    except Exception:
        pass
    # 2) Quick heuristics: single->double quotes, remove trailing commas before } or ]
    cand = candidate.replace("'", '"')
    cand = re.sub(r",\s*([}\]])", r"\1", cand)
    try:
        return json.loads(cand)
    except Exception:
        pass
    # 3) ast.literal_eval as a last structured attempt (can handle Python dicts)
    try:
        return ast.literal_eval(candidate)
    except Exception:
        pass
    # 4) Give up and return the raw extracted string
    return candidate

def parse_and_dump(s):
    obj = _parse_loose_json(_extract_last_json_str(s))
    if isinstance(obj, (dict, list)):
        return json.dumps(obj, ensure_ascii=False)
    return obj

df["extracted_zeroshot_response"] = df["zeroshot_response"].apply(parse_and_dump)
df["extracted_oneshot_response"] = df["oneshot_response"].apply(parse_and_dump)
df.to_excel(OUTPUT_PATH_2, index=False, engine="openpyxl")

In [3]:


# config (edit as needed)
INPUT_PATH = "data/incontext_sinhala.csv"
OUTPUT_PATH_1 = "outputs/incontext_sinhala_gemma.xlsx"
OUTPUT_PATH_2 = "outputs/incontext_sinhala_gemma_extracted.xlsx"
BATCH_SIZE = 32


# load data, model, tokenizer
df = pd.read_csv(INPUT_PATH)

# build prompts (None for empties)
prompts = []
for p in df["zeroshot_prompt"]:
    if pd.isna(p) or str(p).strip() == "":
        prompts.append(None)
    else:
        messages = [
            {
                "role": "system",
                "content": [{"type": "text", "text": SYSTEM_PROMPT}]
            },
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": p}
                ]
            }
        ]

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        prompts.append(prompt)

    
# indices & results placeholder
non_empty_indices = [i for i, p in enumerate(prompts) if p is not None]
results = [""] * len(prompts)

# batch loop (simple)
for start in tqdm(range(0, len(non_empty_indices), BATCH_SIZE), desc="Batched generation"):
    batch_indices = non_empty_indices[start:start + BATCH_SIZE]
    batch_prompts = [prompts[i] for i in batch_indices]
    try:
        outs = llm.generate(batch_prompts, sampling_params=sampling)
        for j, out in enumerate(outs):
            text = tokenizer.decode(out.outputs[0].token_ids, skip_special_tokens=True).strip("\n")
            results[batch_indices[j]] = text
    except Exception as e:
        for i in batch_indices:
            results[i] = f"[ERROR] {type(e).__name__}: {str(e)[:200]}"

# save back to dataframe
df["zeroshot_response"] = results




prompts = []
for p in df["oneshot_prompt"]:
    if pd.isna(p) or str(p).strip() == "":
        prompts.append(None)
    else:
        messages = [
            {
                "role": "system",
                "content": [{"type": "text", "text": SYSTEM_PROMPT}]
            },
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": p}
                ]
            }
        ]

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        prompts.append(prompt)

    
# indices & results placeholder
non_empty_indices = [i for i, p in enumerate(prompts) if p is not None]
results = [""] * len(prompts)

# batch loop (simple)
for start in tqdm(range(0, len(non_empty_indices), BATCH_SIZE), desc="Batched generation"):
    batch_indices = non_empty_indices[start:start + BATCH_SIZE]
    batch_prompts = [prompts[i] for i in batch_indices]
    try:
        outs = llm.generate(batch_prompts, sampling_params=sampling)
        for j, out in enumerate(outs):
            text = tokenizer.decode(out.outputs[0].token_ids, skip_special_tokens=True).strip("\n")
            results[batch_indices[j]] = text
    except Exception as e:
        for i in batch_indices:
            results[i] = f"[ERROR] {type(e).__name__}: {str(e)[:200]}"

# save back to dataframe
df["oneshot_response"] = results


df.to_excel(OUTPUT_PATH_1, index=False, engine="openpyxl")

Batched generation:   0%|          | 0/2 [00:00<?, ?it/s]

Adding requests:   0%|          | 0/32 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/32 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Batched generation:  50%|█████     | 1/2 [00:02<00:02,  2.20s/it]

Adding requests:   0%|          | 0/21 [00:00<?, ?it/s]

...

Batched generation:  50%|█████     | 1/2 [00:02<00:02,  2.65s/it]

Adding requests:   0%|          | 0/21 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/21 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Batched generation: 100%|██████████| 2/2 [00:05<00:00,  2.62s/it]

Batched generation: 100%|██████████| 2/2 [00:05<00:00,  2.63s/it]

In [4]:
# Apply to dataframe (no in-place overwrite until all succeed)
df["extracted_zeroshot_response"] = df["zeroshot_response"].apply(parse_and_dump)
df["extracted_oneshot_response"] = df["oneshot_response"].apply(parse_and_dump)
df.to_excel(OUTPUT_PATH_2, index=False, engine="openpyxl")

In [5]:


# config (edit as needed)
INPUT_PATH = "data/incontext_tamil.csv"
OUTPUT_PATH_1 = "outputs/incontext_tamil_gemma.xlsx"
OUTPUT_PATH_2 = "outputs/incontext_tamil_gemma_extracted.xlsx"
BATCH_SIZE = 32


# load data, model, tokenizer
df = pd.read_csv(INPUT_PATH)

# build prompts (None for empties)
prompts = []
for p in df["zeroshot_prompt"]:
    if pd.isna(p) or str(p).strip() == "":
        prompts.append(None)
    else:
        messages = [
            {
                "role": "system",
                "content": [{"type": "text", "text": SYSTEM_PROMPT}]
            },
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": p}
                ]
            }
        ]

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        prompts.append(prompt)

    
# indices & results placeholder
non_empty_indices = [i for i, p in enumerate(prompts) if p is not None]
results = [""] * len(prompts)

# batch loop (simple)
for start in tqdm(range(0, len(non_empty_indices), BATCH_SIZE), desc="Batched generation"):
    batch_indices = non_empty_indices[start:start + BATCH_SIZE]
    batch_prompts = [prompts[i] for i in batch_indices]
    try:
        outs = llm.generate(batch_prompts, sampling_params=sampling)
        for j, out in enumerate(outs):
            text = tokenizer.decode(out.outputs[0].token_ids, skip_special_tokens=True).strip("\n")
            results[batch_indices[j]] = text
    except Exception as e:
        for i in batch_indices:
            results[i] = f"[ERROR] {type(e).__name__}: {str(e)[:200]}"

# save back to dataframe
df["zeroshot_response"] = results




prompts = []
for p in df["oneshot_prompt"]:
    if pd.isna(p) or str(p).strip() == "":
        prompts.append(None)
    else:
        messages = [
            {
                "role": "system",
                "content": [{"type": "text", "text": SYSTEM_PROMPT}]
            },
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": p}
                ]
            }
        ]

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        prompts.append(prompt)

    
# indices & results placeholder
non_empty_indices = [i for i, p in enumerate(prompts) if p is not None]
results = [""] * len(prompts)

# batch loop (simple)
for start in tqdm(range(0, len(non_empty_indices), BATCH_SIZE), desc="Batched generation"):
    batch_indices = non_empty_indices[start:start + BATCH_SIZE]
    batch_prompts = [prompts[i] for i in batch_indices]
    try:
        outs = llm.generate(batch_prompts, sampling_params=sampling)
        for j, out in enumerate(outs):
            text = tokenizer.decode(out.outputs[0].token_ids, skip_special_tokens=True).strip("\n")
            results[batch_indices[j]] = text
    except Exception as e:
        for i in batch_indices:
            results[i] = f"[ERROR] {type(e).__name__}: {str(e)[:200]}"

# save back to dataframe
df["oneshot_response"] = results


df.to_excel(OUTPUT_PATH_1, index=False, engine="openpyxl")

Batched generation:   0%|          | 0/2 [00:00<?, ?it/s]

Adding requests:   0%|          | 0/32 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/32 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Batched generation:  50%|█████     | 1/2 [00:01<00:01,  1.81s/it]

Adding requests:   0%|          | 0/21 [00:00<?, ?it/s]

...

Batched generation:  50%|█████     | 1/2 [00:01<00:01,  1.87s/it]

Adding requests:   0%|          | 0/21 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/21 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Batched generation: 100%|██████████| 2/2 [00:03<00:00,  1.75s/it]

Batched generation: 100%|██████████| 2/2 [00:03<00:00,  1.77s/it]

In [6]:

# Apply to dataframe (no in-place overwrite until all succeed)
df["extracted_zeroshot_response"] = df["zeroshot_response"].apply(parse_and_dump)
df["extracted_oneshot_response"] = df["oneshot_response"].apply(parse_and_dump)
df.to_excel(OUTPUT_PATH_2, index=False, engine="openpyxl")